In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [2]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )

In [3]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        # network parameters
        hidden_units = 256
        dropout = 0.45
        input_size = 784
        num_labels = 10
        # Define the layers
        self.fc1 = nn.Linear(input_size, hidden_units)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_units, hidden_units)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_units, num_labels)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

In [4]:
model = Model()
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "pytorch",
      "params": {
        "loss": nn.CrossEntropyLoss(),
        "optimizer": optim.Adam(model.parameters(), lr=0.001)

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [5]:
X_train, y_train = get_train_data()
y_train = np.argmax(y_train, axis=1)

In [6]:
rain = Rain(config, model)

2023-07-07 22:19:31,996 [DEBUG] [Rain] Rain is initialized
2023-07-07 22:19:31,997 [DEBUG] [Provisioner] Creating coordinator
2023-07-07 22:19:31,998 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-07 22:19:31,999 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-07 22:19:32,000 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-07 22:19:32,001 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 22:19:32,002 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 22:19:32,004 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [7]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-07 22:19:32,015 [INFO] [Provisioner] provisioner is serving
2023-07-07 22:19:32,016 [DEBUG] [Provisioner] Starting coordinator
2023-07-07 22:19:32,018 [INFO] [Coordinator] coordinator is serving
2023-07-07 22:19:32,019 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-07 22:19:32,025 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 22:19:32,026 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-07 22:19:32,027 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
2023-07-07 22:19:32,029 [DEBUG] [Provisioner] Provision requested the coordinator to get the number of workers
2023-07-07 22:19:32,030 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-07 22:19:32,031 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-07 22:19:32,033 [INFO] [Worker_50151] Worker is running 

Epoch [1/5], Loss: 0.7280, Accuracy: 0.7793
Epoch [1/5], Loss: 0.7216, Accuracy: 0.7802
Epoch [1/5], Loss: 0.7414, Accuracy: 0.7759
Epoch [2/5], Loss: 0.2997, Accuracy: 0.9127
Epoch [2/5], Loss: 0.2998, Accuracy: 0.9100
Epoch [2/5], Loss: 0.3172, Accuracy: 0.9048
Epoch [3/5], Loss: 0.2306, Accuracy: 0.9327
Epoch [3/5], Loss: 0.2301, Accuracy: 0.9295
Epoch [3/5], Loss: 0.2423, Accuracy: 0.9245
Epoch [4/5], Loss: 0.1890, Accuracy: 0.9429
Epoch [4/5], Loss: 0.1849, Accuracy: 0.9418
Epoch [4/5], Loss: 0.1993, Accuracy: 0.9397


2023-07-07 22:19:42,332 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-07 22:19:42,333 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-07 22:19:42,402 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-07 22:19:42,408 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-07 22:19:42,409 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
2023-07-07 22:19:42,424 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-07 22:19:42,425 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
2023-07-07 22:19:42,466 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_1_trained.pkl from w

Epoch [5/5], Loss: 0.1635, Accuracy: 0.9507
sending data to divider
Epoch [5/5], Loss: 0.1628, Accuracy: 0.9499
sending data to divider
Epoch [5/5], Loss: 0.1650, Accuracy: 0.9498
sending data to divider


2023-07-07 22:19:42,516 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-07 22:19:42,517 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
2023-07-07 22:19:42,517 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
2023-07-07 22:19:42,519 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration 2 to worker 1
2023-07-07 22:19:42,520 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration 2 to worker 2
2023-07-07 22:19:42,521 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration 2 to worker 3
2023-07-07 22:19:42,522 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/1.pkl to worker1
2023-07-07 22:19:42,523 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/2.pkl to worker2
2023-07-07 22:19:42,523 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/3.pkl to worker3
2023-07-07 22:19:42,591 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
2023-07-07

...

Epoch [5/5], Loss: 0.1132, Accuracy: 0.9640
Epoch [5/5], Loss: 0.1025, Accuracy: 0.9668
sending data to divider
Epoch [5/5], Loss: 0.1056, Accuracy: 0.9673
sending data to divider
sending data to divider


2023-07-07 22:19:49,542 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
2023-07-07 22:19:49,543 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker3
2023-07-07 22:19:49,549 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 3


Epoch [1/5], Loss: 0.1262, Accuracy: 0.9625
Epoch [1/5], Loss: 0.1293, Accuracy: 0.9608
Epoch [1/5], Loss: 0.1201, Accuracy: 0.9629
Epoch [2/5], Loss: 0.1121, Accuracy: 0.9664
Epoch [2/5], Loss: 0.1101, Accuracy: 0.9660
Epoch [2/5], Loss: 0.1016, Accuracy: 0.9688
Epoch [3/5], Loss: 0.0952, Accuracy: 0.9709
Epoch [3/5], Loss: 0.0997, Accuracy: 0.9697
Epoch [3/5], Loss: 0.0941, Accuracy: 0.9712
Epoch [4/5], Loss: 0.0892, Accuracy: 0.9716
Epoch [4/5], Loss: 0.0858, Accuracy: 0.9732
Epoch [4/5], Loss: 0.0866, Accuracy: 0.9728


2023-07-07 22:19:56,309 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-07 22:19:56,310 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-07 22:19:56,311 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
2023-07-07 22:19:56,313 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
2023-07-07 22:19:56,332 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-07 22:19:56,333 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-07 22:19:56,369 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-07 22:19:56,373 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from w

Epoch [5/5], Loss: 0.0823, Accuracy: 0.9740
Epoch [5/5], Loss: 0.0768, Accuracy: 0.9764
sending data to divider
sending data to divider
Epoch [5/5], Loss: 0.0765, Accuracy: 0.9756
sending data to divider


In [8]:
def evaluate_model(model, X_test, y_test, batch_size):
    # Convert numpy arrays to PyTorch tensors
    X_test = torch.from_numpy(X_test).float()
    y_test = torch.from_numpy(y_test).long()

    # Create a TensorDataset
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    total_correct = 0
    total_samples = 0
    
    for i, (data, labels) in enumerate(test_loader):
        # Forward pass
        outputs = model(data)

        # Compute training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)

    return total_correct / total_samples

In [9]:
X_test, y_test = get_test_data()
y_test = np.argmax(y_test, axis=1)
acc = evaluate_model(model, X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))


Test accuracy: 97.0%
